In [ ]:
import pandas as pd
import numpy as np
import re
import string
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

def load_and_preprocess_data():
    home_dir = os.path.expanduser("~")
    file_path = os.path.join(home_dir, "Downloads", "tweet_emotions.csv")
    
    if not os.path.exists(file_path):
        alternative_path = os.path.join(home_dir, "downloads", "tweet_emotions.csv")
        if os.path.exists(alternative_path):
            file_path = alternative_path
        else:
            current_dir_file = "tweet_emotions.csv"
            if os.path.exists(current_dir_file):
                file_path = current_dir_file
            else:
                print(f"System Lookups Diagnostic Checking: Checked paths in Downloads and current workspace.")
                raise FileNotFoundError("System error: 'tweet_emotions.csv' file dynamically track avvaledu. Please check if the file is in your Downloads folder.")
                
    print(f"--- Dataset Found At: {file_path} ---")
    print("--- Loading Dataset ---")
    df = pd.read_csv(file_path)
    df = df[['content', 'sentiment']].dropna()
    
    def advance_clean_text(text):
        text = str(text)
        text = re.sub(r'@[A-Za-z0-9_]+', '', text)
        text = re.sub(r'https?://\S+|www\.\S+', '', text)
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\d+', '', text)
        return text.lower().strip()
        
    df['clean_content'] = df['content'].apply(advance_clean_text)
    df = df[df['clean_content'] != '']
    
    print(f"Successfully loaded and cleaned {len(df)} records.")
    print(f"Detected Emotions/Sentiments: {list(df['sentiment'].unique())}\n")
    return df

def main():
    data = load_and_preprocess_data()
    
    X_train, X_test, y_train, y_test = train_test_split(
        data['clean_content'], 
        data['sentiment'], 
        test_size=0.2, 
        random_state=42, 
        stratify=data['sentiment']
    )
    
    print("--- Converting Text to Vector Features ---")
    vectorizer = TfidfVectorizer(max_features=12000, stop_words='english', sublinear_tf=True, ngram_range=(1, 2))
    X_train_vectors = vectorizer.fit_transform(X_train)
    X_test_vectors = vectorizer.transform(X_test)
    
    print("--- Training Logistic Regression Model ---")
    model = LogisticRegression(max_iter=1000, solver='lbfgs', class_weight='balanced', random_state=42)
    model.fit(X_train_vectors, y_train)
    
    print("--- Evaluating Model Performance ---")
    predictions = model.predict(X_test_vectors)
    accuracy = accuracy_score(y_test, predictions)
    print(f"Overall Validation Accuracy: {accuracy * 100:.2f}%\n")
    print("Classification Metrics Matrix:")
    print(classification_report(y_test, predictions, zero_division=0))
    
    print("====================================================")
    print("   AI-Powered Emotion Prediction Prompt Enabled     ")
    print("   Type a sentence below to test. Enter 'exit' to quit. ")
    print("====================================================")
    
    while True:
        user_input = input("\nEnter text: ").strip()
        if user_input.lower() == 'exit':
            print("Exiting pipeline setup. Goodbye!")
            break
        if not user_input:
            continue
            
        text_regex = re.sub(r'@[A-Za-z0-9_]+', '', user_input)
        text_regex = re.sub(r'https?://\S+|www\.\S+', '', text_regex)
        cleaned_input = text_regex.translate(str.maketrans('', '', string.punctuation)).lower().strip()
        
        input_vector = vectorizer.transform([cleaned_input])
        prediction = model.predict(input_vector)[0]
        probabilities = model.predict_proba(input_vector)[0]
        confidence_score = np.max(probabilities)
        
        print(f"Predicted Underlying Emotion: [{prediction.upper()}] (Confidence: {confidence_score:.2f})")

if __name__ == "__main__":
    main()

--- Dataset Found At: C:\Users\deepa\Downloads\tweet_emotions.csv ---
--- Loading Dataset ---
Successfully loaded and cleaned 39901 records.
Detected Emotions/Sentiments: ['empty', 'sadness', 'enthusiasm', 'neutral', 'worry', 'surprise', 'love', 'fun', 'hate', 'happiness', 'boredom', 'relief', 'anger']

--- Converting Text to Vector Features ---
--- Training Logistic Regression Model ---
--- Evaluating Model Performance ---
Overall Validation Accuracy: 24.71%

Classification Metrics Matrix:
              precision    recall  f1-score   support

       anger       0.00      0.00      0.00        22
     boredom       0.04      0.17      0.07        36
       empty       0.04      0.14      0.06       162
  enthusiasm       0.03      0.13      0.06       152
         fun       0.13      0.25      0.17       355
   happiness       0.34      0.24      0.28      1042
        hate       0.20      0.40      0.26       265
        love       0.46      0.45      0.45       768
     neutral     


Enter text:  i am going to home today


Predicted Underlying Emotion: [NEUTRAL] (Confidence: 0.16)



Enter text:  i aam so haappy todaay


Predicted Underlying Emotion: [NEUTRAL] (Confidence: 0.16)



Enter text:  we are going cinemaa today


Predicted Underlying Emotion: [NEUTRAL] (Confidence: 0.15)
